# Obtendo Dados do Supabase
Neste projeto a coleta de dados para criação do dataset foi realizada através de um formulário e salva em um banco de dados no Supabase.

In [2]:
!pip install supabase pandas
!apt-get -qq install ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [supabase]


### Exporta as gravações do Supabase para dados/audios/*.wav e gera dados/metadata.csv

> IMPORTANTE! Adicione as chaves ao notebook, SUPABASE_URL e SUPABASE_SERVICE_KEY

In [5]:
import os, subprocess, tempfile
import pandas as pd
from supabase import create_client

try:
    from google.colab import userdata
    URL, KEY = userdata.get("SUPABASE_URL"), userdata.get("SUPABASE_SERVICE_KEY")
except ImportError:
    URL, KEY = os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_KEY"]

sb = create_client(URL, KEY)
BUCKET = "voice-recordings"
OUT_AUDIO, OUT_META = "dados/audios", "dados/metadata.csv"
os.makedirs(OUT_AUDIO, exist_ok=True)
EXT = {"audio/webm": ".webm", "audio/mp4": ".m4a", "audio/ogg": ".ogg", "audio/wav": ".wav", "audio/x-wav": ".wav"}

# 1. Metadados: só gravações originais e completas (sem nome do participante)
linhas, inicio, passo = [], 0, 1000
while True:
    resp = (sb.table("recordings")
              .select("id,object_path,repetition,mime_type,duration_ms,participant_id,"
                      "phrases(label),participants(sex,age,completed_at)")
              .eq("status", "complete").is_("source_recording_id", "null")
              .range(inicio, inicio + passo - 1).execute())
    linhas += resp.data
    if len(resp.data) < passo:
        break
    inicio += passo

# Pseudônimo curto por participante (P001, P002...) no lugar do UUID
ids = sorted({r["participant_id"] for r in linhas})
apelido = {pid: f"P{i+1:03d}" for i, pid in enumerate(ids)}

meta = []
for r in linhas:
    loc, rot = apelido[r["participant_id"]], r["phrases"]["label"]
    nome = f"{loc}_{rot}_r{r['repetition']}.wav"
    destino = os.path.join(OUT_AUDIO, nome)

    # 2. Baixa o áudio e converte para WAV mono 16 kHz
    if not os.path.exists(destino):
        dados = sb.storage.from_(BUCKET).download(r["object_path"])
        with tempfile.NamedTemporaryFile(suffix=EXT.get(r["mime_type"], ".bin")) as tmp:
            tmp.write(dados); tmp.flush()
            subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", tmp.name,
                            "-ac", "1", "-ar", "16000", destino], check=True)

    p = r["participants"]
    meta.append(dict(arquivo=nome, locutor=loc, rotulo=rot, tipo="comando",
                     repeticao=r["repetition"], sexo=p["sex"], idade=p["age"],
                     participante_completo=p["completed_at"] is not None,
                     formato_original=r["mime_type"], duracao_ms=r["duration_ms"]))

df = pd.DataFrame(meta).sort_values(["locutor", "rotulo", "repeticao"])
df.to_csv(OUT_META, index=False)
print(f"{len(df)} gravações de {df.locutor.nunique()} participantes")
print(df.groupby("rotulo").size())
# Depois, acrescente os negativos (Common Voice / frases sem comando) ao mesmo CSV,
# com tipo="negativo", rotulo="fora" e locutor = client_id do Common Voice.

168 gravações de 14 participantes
rotulo
abrir_cortina          28
acender_luz            28
apagar_luz             28
desligar_ventilador    28
fechar_cortina         28
ligar_ventilador       28
dtype: int64


# Carregando Dados Negativos do Common Voice pt-br

Os dados negativos serão utilizados em um dos testes como exemplos que o modelo não deve reconhecer como comandos.

Montagem do Drive

In [10]:
from google.colab import drive; drive.mount('/content/drive')
!apt-get -qq install ffmpeg

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Montagem do Dataset com Negativas


In [ ]:
import os, subprocess
import pandas as pd

CV_DIR = "/content/drive/MyDrive/SEU_CAMIHNHO_PARA_OS_DADOS"   # pasta que contém train.tsv, dev.tsv, test.tsv e clips/
OUT_AUDIO, META = "dados/audios", "dados/metadata.csv"
N_CLIPES = 600            # ~ o mesmo número de áudios de comando
MAX_POR_LOCUTOR = 3       # evita que poucos locutores dominem a classe negativa
SEED = 42

# Frases que lembram os comandos ficam de fora: senão não seriam "fora do escopo"
PALAVRAS = r"\b(?:luz|luzes|ventilador|cortina|cortinas|acend\w*|apag\w*|lig\w*|deslig\w*|abr\w*|fech\w*)\b"

# 1. Junta as listas (a divisão oficial do Common Voice não é usada: a nossa é por locutor)
tabelas = []
for nome in ["train.tsv", "dev.tsv", "test.tsv", "validated.tsv"]:
    caminho = os.path.join(CV_DIR, nome)
    if os.path.exists(caminho):
        tabelas.append(pd.read_csv(caminho, sep="\t", quoting=3, low_memory=False))
if not tabelas:
    raise FileNotFoundError(f"Nenhum .tsv encontrado em {CV_DIR}")
val = pd.concat(tabelas).drop_duplicates("path")

# Só clipes que existem de fato na pasta clips/
val = val[val["path"].apply(lambda p: os.path.exists(os.path.join(CV_DIR, "clips", p)))]
val = val[~val["sentence"].str.lower().str.contains(PALAVRAS, regex=True, na=False)]
amostra = (val.sample(frac=1, random_state=SEED)
              .groupby("client_id").head(MAX_POR_LOCUTOR)
              .head(N_CLIPES))
print(f"{len(val)} clipes disponíveis após o filtro; selecionados {len(amostra)} "
      f"de {amostra.client_id.nunique()} locutores")

# 2. Converte os escolhidos para WAV mono 16 kHz, com locutor pseudonimizado
apelido = {c: f"C{i+1:04d}" for i, c in enumerate(sorted(amostra["client_id"].unique()))}
os.makedirs(OUT_AUDIO, exist_ok=True)
novos = []
for _, r in amostra.iterrows():
    loc = apelido[r["client_id"]]
    nome_wav = f"{loc}_{os.path.splitext(r['path'])[0]}.wav"
    destino = os.path.join(OUT_AUDIO, nome_wav)
    if not os.path.exists(destino):
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", os.path.join(CV_DIR, "clips", r["path"]),
                        "-ac", "1", "-ar", "16000", destino], check=True)
    novos.append(dict(arquivo=nome_wav, locutor=loc, rotulo="fora", tipo="negativo",
                      origem="common_voice", texto=r["sentence"]))

# 3. Junta à planilha dos comandos (sem duplicar se rodar de novo)
meta = pd.read_csv(META)
if "origem" not in meta:
    meta["origem"] = "gravacao_grupo"
meta = meta[meta["origem"] != "common_voice"]
meta = pd.concat([meta, pd.DataFrame(novos)], ignore_index=True)
meta.to_csv(META, index=False)
print(meta.groupby(["tipo", "origem"]).agg(audios=("arquivo", "size"), locutores=("locutor", "nunique")))

1942 clipes disponíveis após o filtro; selecionados 165 de 55 locutores
                         audios  locutores
tipo     origem                           
comando  gravacao_grupo     168         14
negativo common_voice       165         55


## Download dos Ruídos MUSAN

In [18]:
!mkdir -p dados
!wget -qO- http://www.openslr.org/resources/17/musan.tar.gz | tar -xz -C dados musan/noise
!find dados/musan/noise -name "*.wav" | wc -l

930


# Salvamento dos Dados

In [ ]:
!cd /content && zip -qr /content/drive/MyDrive/""SEU CAMINHO PARA OS DADOS"" dados